In [ ]:
"""
Automated Subpixel Quantum Dot & Alignment Marker Positioning Pipeline
Author: Jakub Olik
Description:
    Locates reference alignment markers and self-assembled quantum dots (QDs) 
    in cryogenic photoluminescence (PL) images with subpixel precision using 
    2D Gaussian fitting and image moments. Automatically processes batch images 
    and exports positioning coordinates to CSV and TXT reports sorted by QD intensity.
"""

import csv
import glob
import os
import cv2
import numpy as np
from scipy.optimize import curve_fit

# ==============================================================================
# USER CONFIGURATION / PARAMETERS
# ==============================================================================

# Physical size of the square alignment marker grid [micrometers]
GRID_SIZE_UM = 80.0

# Detection sensitivity threshold (multiplied by standard deviation of noise after background subtraction)
# Typical values: 2.5 - 3.5 (lower detects dimmer QDs, higher avoids background noise)
THRESHOLD_SIGMA = 3.0

# Margin outside the grid boundaries to inspect for additional QDs [micrometers]
MARGIN_UM = 15.0

# Minimum Euclidean distance between detected peaks to merge saturation duplicates [pixels]
MIN_DIST_NMS_PX = 5.0

# Approximate microscope pixel scale [um/px] used ONLY for initial marker candidate filtering
# (The exact subpixel scale is dynamically calculated for each image)
APPROX_PIXEL_SCALE_UM_PER_PX = 0.246

# Radius of alignment marker circles [pixels]
MARKER_RADIUS_PX = 17

# Directory containing images to process ('.' = current directory)
INPUT_FOLDER = "."

# ==============================================================================
# CORE ALGORITHMS
# ==============================================================================

def gaussian_2d(xy, A, x0, y0, sigma_x, sigma_y, theta, offset):
    """2D Gaussian surface model for point spread function (PSF) fitting."""
    x, y = xy
    a = (np.cos(theta) ** 2) / (2 * sigma_x**2) + (np.sin(theta) ** 2) / (2 * sigma_y**2)
    b = -(np.sin(2 * theta)) / (4 * sigma_x**2) + (np.sin(2 * theta)) / (4 * sigma_y**2)
    c = (np.sin(theta) ** 2) / (2 * sigma_x**2) + (np.cos(theta) ** 2) / (2 * sigma_y**2)
    
    g = offset + A * np.exp(
        -(
            a * ((x - x0) ** 2)
            + 2 * b * (x - x0) * (y - y0)
            + c * ((y - y0) ** 2)
        )
    )
    return g.ravel()


def refine_marker_subpixel(gray, x_c, y_c, r=20):
    """Calculates subpixel center of mass for markers using image moments after Otsu thresholding."""
    h, w = gray.shape
    x1, x2 = max(0, int(x_c - r)), min(w, int(x_c + r + 1))
    y1, y2 = max(0, int(y_c - r)), min(h, int(y_c + r + 1))
    roi = gray[y1:y2, x1:x2]

    _, th = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    M = cv2.moments(th)
    if M["m00"] != 0:
        return x1 + M["m10"] / M["m00"], y1 + M["m01"] / M["m00"]
    return float(x_c), float(y_c)


def fit_qd_gaussian(gray, x_c, y_c, r=7):
    """Fits an anisotropic 2D Gaussian profile to localize a single QD with subpixel accuracy."""
    h, w = gray.shape
    x1, x2 = max(0, int(x_c - r)), min(w, int(x_c + r + 1))
    y1, y2 = max(0, int(y_c - r)), min(h, int(y_c + r + 1))

    roi = gray[y1:y2, x1:x2].astype(np.float64)
    y_g, x_g = np.mgrid[y1:y2, x1:x2]

    bg = np.min(roi)
    A = np.max(roi) - bg
    p0 = [A, x_c, y_c, 1.5, 1.5, 0.0, bg]
    bounds = ([0, x1, y1, 0.5, 0.5, -np.pi, 0], [np.inf, x2, y2, r, r, np.pi, np.inf])

    try:
        popt, _ = curve_fit(gaussian_2d, (x_g, y_g), roi.ravel(), p0=p0, bounds=bounds)
        if popt[3] > 4.5 or popt[4] > 4.5:
            return None
        return popt[1], popt[2]
    except Exception:
        return None


def create_circle_template(radius=17):
    """Generates a synthetic zero-mean blurred circular template for marker cross-correlation."""
    size = radius * 2 + 15
    template = np.zeros((size, size), dtype=np.float32)
    center = size // 2
    cv2.circle(template, (center, center), radius, 1.0, -1)
    template = cv2.GaussianBlur(template, (9, 9), 2.5)
    template -= np.mean(template)
    template /= np.std(template) + 1e-6
    return template


def detect_grid_markers(gray, grid_size_um=80.0, approx_scale_um_per_px=0.246, marker_radius_px=17):
    """
    Detects markers via cross-correlation template matching and identifies
    the bottom-left (0,0) and top-right (grid_size, grid_size) pair based on geometry.
    """
    expected_diag_um = np.sqrt(2.0) * grid_size_um
    expected_diag_px = expected_diag_um / approx_scale_um_per_px
    tolerance_px = expected_diag_px * 0.12

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray).astype(np.float32)
    bg_smoothed = cv2.medianBlur(enhanced.astype(np.uint8), 7).astype(np.float32)

    template = create_circle_template(radius=marker_radius_px)
    corr_map = cv2.matchTemplate(bg_smoothed, template, cv2.TM_CCOEFF_NORMED)

    pad_h = (bg_smoothed.shape[0] - corr_map.shape[0]) // 2
    pad_w = (bg_smoothed.shape[1] - corr_map.shape[1]) // 2
    padded_corr = np.pad(corr_map, ((pad_h, pad_h), (pad_w, pad_w)), mode="constant")

    thresh = np.percentile(corr_map, 99.0)
    kernel = np.ones((25, 25), np.uint8)
    dilated = cv2.dilate(padded_corr, kernel)
    peaks = (padded_corr == dilated) & (padded_corr > thresh)

    pts = np.argwhere(peaks)
    if len(pts) < 2:
        thresh = np.percentile(corr_map, 98.0)
        peaks = (padded_corr == dilated) & (padded_corr > thresh)
        pts = np.argwhere(peaks)

    pts = np.array([[p[1], p[0]] for p in pts])
    if len(pts) < 2:
        raise RuntimeError(f"Detected only {len(pts)} marker candidate(s). Verify marker size or contrast.")

    best_pair = None
    min_dist_err = float("inf")

    for i in range(len(pts)):
        for j in range(len(pts)):
            if i == j:
                continue
            p_bl, p_tr = pts[i], pts[j]
            dx = p_tr[0] - p_bl[0]
            dy = p_bl[1] - p_tr[1]

            if dx > (expected_diag_px * 0.3) and dy > (expected_diag_px * 0.3):
                dist = np.sqrt(dx**2 + dy**2)
                aspect = dx / dy
                if abs(dist - expected_diag_px) < tolerance_px and 0.80 < aspect < 1.25:
                    err = abs(dist - expected_diag_px) + abs(dx - dy) * 1.5
                    if err < min_dist_err:
                        min_dist_err = err
                        best_pair = (p_bl, p_tr)

    if best_pair is None:
        raise RuntimeError(
            f"Detected {len(pts)} candidates, but none match the {expected_diag_px:.1f} px diagonal "
            f"for a {grid_size_um}x{grid_size_um} um grid."
        )

    m0_conf = padded_corr[best_pair[0][1], best_pair[0][0]]
    m80_conf = padded_corr[best_pair[1][1], best_pair[1][0]]
    marker_confidence = float((m0_conf + m80_conf) / 2.0)

    m0 = refine_marker_subpixel(gray, best_pair[0][0], best_pair[0][1], r=marker_radius_px + 3)
    m80 = refine_marker_subpixel(gray, best_pair[1][0], best_pair[1][1], r=marker_radius_px + 3)
    return m0, m80, marker_confidence


def find_all_qds_auto(gray, m0, m80, grid_size_um=80.0, threshold_sigma=3.0, margin_um=15.0, min_dist_px=5.0):
    """Detects emission peaks, fits 2D Gaussians, and removes duplicates via Non-Maximum Suppression (NMS)."""
    mask = np.ones_like(gray, dtype=bool)
    for m in [m0, m80]:
        cv2.circle(mask.view(np.uint8), (int(m[0]), int(m[1])), 25, 0, -1)

    # Local Background Subtraction to handle non-uniform illumination and contrast variations
    gray_f = gray.astype(np.float32)
    bg_smooth = cv2.GaussianBlur(gray_f, (31, 31), 10)
    diff = gray_f - bg_smooth
    diff[diff < 0] = 0

    diff_mean = np.mean(diff[mask])
    diff_std = np.std(diff[mask])
    threshold_val = diff_mean + threshold_sigma * diff_std

    kernel = np.ones((5, 5), np.uint8)
    dilated = cv2.dilate(diff, kernel)
    peaks = (diff == dilated) & (diff > threshold_val) & mask
    peak_coords = np.argwhere(peaks)

    sx = grid_size_um / (m80[0] - m0[0])
    sy = grid_size_um / (m0[1] - m80[1])

    x_min, x_max = -margin_um, grid_size_um + margin_um
    y_min, y_max = -margin_um, grid_size_um + margin_um

    raw_qds = []
    snr_list = []

    for y_rough, x_rough in peak_coords:
        res = fit_qd_gaussian(gray, x_rough, y_rough)
        if res is not None:
            sub_x, sub_y = res
            x_um = (sub_x - m0[0]) * sx
            y_um = (m0[1] - sub_y) * sy

            if x_min <= x_um <= x_max and y_min <= y_um <= y_max:
                is_inside = (0.0 <= x_um <= grid_size_um) and (0.0 <= y_um <= grid_size_um)
                intensity_raw = int(gray[y_rough, x_rough])
                intensity_diff = float(diff[y_rough, x_rough])
                snr = intensity_diff / (diff_std + 1e-6)

                raw_qds.append({
                    "px": (sub_x, sub_y),
                    "um": (x_um, y_um),
                    "intensity": intensity_raw,
                    "snr": round(snr, 2),
                    "inside_grid": "YES" if is_inside else "NO",
                })

    raw_qds.sort(key=lambda d: d["intensity"], reverse=True)

    unique_qds = []
    for candidate in raw_qds:
        c_x, c_y = candidate["px"]
        is_duplicate = False
        for accepted in unique_qds:
            a_x, a_y = accepted["px"]
            dist = np.hypot(c_x - a_x, c_y - a_y)
            if dist < min_dist_px:
                is_duplicate = True
                break

        if not is_duplicate:
            unique_qds.append(candidate)
            snr_list.append(candidate["snr"])

    for qd in unique_qds:
        qd["px"] = (round(qd["px"][0], 3), round(qd["px"][1], 3))
        qd["um"] = (round(qd["um"][0], 3), round(qd["um"][1], 3))

    mean_snr = float(np.mean(snr_list)) if snr_list else 0.0
    return unique_qds, (sx, sy), mean_snr


def save_results_to_files(base_name, m0, m80, scales, qds, grid_size_um, marker_conf, mean_snr):
    """Exports localized emitter data to CSV and formatted TXT report files sorted by intensity descending."""
    csv_filename = f"{base_name}_qds.csv"
    txt_filename = f"{base_name}_report.txt"

    # Gwarantowane sortowanie malejąco po jasności (Intensity)
    qds_sorted = sorted(qds, key=lambda d: d["intensity"], reverse=True)

    # Zapis do CSV
    with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Px_X", "Px_Y", "Coord_X_um", "Coord_Y_um", "Inside_Grid", "Intensity", "SNR"])
        for idx, qd in enumerate(qds_sorted):
            writer.writerow([
                idx + 1,
                qd["px"][0],
                qd["px"][1],
                qd["um"][0],
                qd["um"][1],
                qd["inside_grid"],
                qd["intensity"],
                qd["snr"]
            ])

    # Zapis do raportu TXT
    with open(txt_filename, mode="w", encoding="utf-8") as f:
        f.write("=== DETERMINISTIC SUBPIXEL POSITIONING REPORT ===\n")
        f.write(f"Grid Dimensions:    {grid_size_um:.1f} x {grid_size_um:.1f} um\n")
        f.write(f"Marker Confidence:  {marker_conf:.3f} (Scale: 0.0 - 1.0)\n")
        f.write(f"Mean QD SNR:        {mean_snr:.2f}\n")
        f.write(f"Total detected QDs: {len(qds_sorted)}\n\n")
        f.write(f"Marker (0,0):   px = ({m0[0]:.3f}, {m0[1]:.3f})\n")
        f.write(f"Marker ({grid_size_um:.0f},{grid_size_um:.0f}): px = ({m80[0]:.3f}, {m80[1]:.3f})\n")
        f.write(f"Scale:          sx = {scales[0]:.6f} um/px, sy = {scales[1]:.6f} um/px\n\n")
        
        f.write(f"{'ID':<4} | {'Pixels (X, Y)':<18} | {'Position (X, Y) [um]':<30} | {'Inside Grid':<11} | {'Intensity':<9} | {'SNR'}\n")
        f.write("-" * 90 + "\n")
        for idx, qd in enumerate(qds_sorted):
            px_str = f"({qd['px'][0]:.2f}, {qd['px'][1]:.2f})"
            um_str = f"X = {qd['um'][0]:6.2f}, Y = {qd['um'][1]:6.2f}"
            f.write(f"#{idx+1:<3} | {px_str:<18} | {um_str:<30} | {qd['inside_grid']:<11} | {qd['intensity']:<9} | {qd['snr']:.1f}\n")


def process_single_image(
    image_path,
    grid_size_um=GRID_SIZE_UM,
    threshold_sigma=THRESHOLD_SIGMA,
    margin_um=MARGIN_UM,
    min_dist_px=MIN_DIST_NMS_PX,
    approx_scale_um_per_px=APPROX_PIXEL_SCALE_UM_PER_PX,
    marker_radius_px=MARKER_RADIUS_PX,
):
    """Executes the subpixel localization pipeline on an individual image file."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Failed to read image file: {image_path}")

    m0, m80, marker_conf = detect_grid_markers(
        img,
        grid_size_um=grid_size_um,
        approx_scale_um_per_px=approx_scale_um_per_px,
        marker_radius_px=marker_radius_px,
    )

    qds, (sx, sy), mean_snr = find_all_qds_auto(
        img,
        m0,
        m80,
        grid_size_um=grid_size_um,
        threshold_sigma=threshold_sigma,
        margin_um=margin_um,
        min_dist_px=min_dist_px,
    )

    base_name = os.path.splitext(image_path)[0]
    save_results_to_files(base_name, m0, m80, (sx, sy), qds, grid_size_um, marker_conf, mean_snr)

    in_grid = sum(1 for qd in qds if qd["inside_grid"] == "YES")
    out_grid = len(qds) - in_grid

    return {
        "file": os.path.basename(image_path),
        "total_qds": len(qds),
        "in_grid": in_grid,
        "out_grid": out_grid,
        "marker_conf": marker_conf,
        "mean_snr": mean_snr,
        "scale": (sx + sy) / 2.0,
    }


def process_all_images_in_folder(
    folder_path=INPUT_FOLDER,
    grid_size_um=GRID_SIZE_UM,
    threshold_sigma=THRESHOLD_SIGMA,
    margin_um=MARGIN_UM,
    min_dist_px=MIN_DIST_NMS_PX,
    approx_scale_um_per_px=APPROX_PIXEL_SCALE_UM_PER_PX,
    marker_radius_px=MARKER_RADIUS_PX,
):
    """Processes all .png images present in the given folder."""
    png_files = sorted(glob.glob(os.path.join(folder_path, "*.png")))
    if not png_files:
        print(f"No .png images found in target directory: {os.path.abspath(folder_path)}")
        return

    print(f"Starting batch analysis of {len(png_files)} image(s)...\n")
    summary_results = []

    for idx, img_path in enumerate(png_files, start=1):
        filename = os.path.basename(img_path)
        try:
            res = process_single_image(
                img_path,
                grid_size_um=grid_size_um,
                threshold_sigma=threshold_sigma,
                margin_um=margin_um,
                min_dist_px=min_dist_px,
                approx_scale_um_per_px=approx_scale_um_per_px,
                marker_radius_px=marker_radius_px,
            )
            summary_results.append(res)
            print(
                f"[{idx}/{len(png_files)}] {filename:<12} | QDs: {res['total_qds']:<3} "
                f"(IN: {res['in_grid']:<2}, OUT: {res['out_grid']:<2}) | "
                f"Marker Conf: {res['marker_conf']:.2f} | Mean SNR: {res['mean_snr']:.1f}"
            )
        except Exception as e:
            print(f"[{idx}/{len(png_files)}] {filename:<12} | ERROR: {e}")

    print("\n" + "=" * 80)
    print("BATCH PROCESSING SUMMARY")
    print("=" * 80)
    print(f"{'Filename':<15} | {'Total QDs':<10} | {'Inside':<8} | {'Outside':<8} | {'Marker Conf':<12} | {'Mean SNR'}")
    print("-" * 80)
    for r in summary_results:
        print(
            f"{r['file']:<15} | {r['total_qds']:<10} | {r['in_grid']:<8} | "
            f"{r['out_grid']:<8} | {r['marker_conf']:<12.3f} | {r['mean_snr']:.2f}"
        )
    print("=" * 80)


if __name__ == "__main__":
    process_all_images_in_folder()

Starting batch analysis of 4 image(s)...

[1/4] 0202.png     | QDs: 273 (IN: 150, OUT: 123) | Marker Conf: 0.93 | Mean SNR: 5.7
[2/4] 0203.png     | QDs: 68  (IN: 42, OUT: 26) | Marker Conf: 0.91 | Mean SNR: 9.5
[3/4] 0207.png     | QDs: 934 (IN: 510, OUT: 424) | Marker Conf: 0.62 | Mean SNR: 5.2
[4/4] 0211.png     | QDs: 88  (IN: 46, OUT: 42) | Marker Conf: 0.66 | Mean SNR: 5.2

BATCH PROCESSING SUMMARY
Filename        | Total QDs  | Inside   | Outside  | Marker Conf  | Mean SNR
--------------------------------------------------------------------------------
0202.png        | 273        | 150      | 123      | 0.926        | 5.68
0203.png        | 68         | 42       | 26       | 0.910        | 9.54
0207.png        | 934        | 510      | 424      | 0.621        | 5.18
0211.png        | 88         | 46       | 42       | 0.663        | 5.25
